In [1]:
import numpy as np
import pandas as pd
import matplotlib as pyplot
import seaborn as sns


In [2]:
from google.colab import files
uploaded = files.upload()

Saving penguins.csv to penguins.csv


In [4]:
df = pd.read_csv('penguins.csv')

In [5]:
df

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,39.1,18.7,181.0,3750.0,MALE
1,39.5,17.4,186.0,3800.0,FEMALE
2,40.3,18.0,195.0,3250.0,FEMALE
3,NaN,NaN,NaN,NaN,NaN
4,36.7,19.3,193.0,3450.0,FEMALE
...,...,...,...,...,...
339,NaN,NaN,NaN,NaN,NaN
340,46.8,14.3,215.0,4850.0,FEMALE
341,50.4,15.7,222.0,5750.0,MALE
342,45.2,14.8,212.0,5200.0,FEMALE


In [7]:
display(df.describe())

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
count,342.000000,342.000000,342.000000,342.000000
mean,43.921930,17.151170,214.014620,4201.754386
std,5.459584,1.974793,260.558057,801.954536
min,32.100000,13.100000,-132.000000,2700.000000
25%,39.225000,15.600000,190.000000,3550.000000
50%,44.450000,17.300000,197.000000,4050.000000
75%,48.500000,18.700000,213.750000,4750.000000
max,59.600000,21.500000,5000.000000,6300.000000


In [8]:
df_cleaned = df.drop_duplicates()
display(f"Shape of DataFrame before dropping duplicates: {df.shape}")
display(f"Shape of DataFrame after dropping duplicates: {df_cleaned.shape}")

'Shape of DataFrame before dropping duplicates: (344, 5)'

'Shape of DataFrame after dropping duplicates: (343, 5)'

In [10]:
df = df_cleaned.copy()

### Data Preprocessing for KNN

In [11]:
# Check for missing values
display(df.isnull().sum())

,0
culmen_length_mm,1
culmen_depth_mm,1
flipper_length_mm,1
body_mass_g,1
sex,8


In [12]:
df.dropna(subset=['sex'], inplace=True)

for col in ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']:
    if df[col].isnull().any():
        df[col].fillna(df[col].mean(), inplace=True)

display(f"Missing values after imputation:\n{df.isnull().sum()}")

'Missing values after imputation:\nculmen_length_mm     0\nculmen_depth_mm      0\nflipper_length_mm    0\nbody_mass_g          0\nsex                  0\ndtype: int64'

In [13]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

le = LabelEncoder()
df['sex_encoded'] = le.fit_transform(df['sex'])
display(f"Original 'sex' categories: {le.classes_}")
display(f"Encoded 'sex' categories: {df['sex_encoded'].unique()}")

X = df[['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']]
y = df['sex_encoded']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

display(f"Shape of X_train: {X_train.shape}")
display(f"Shape of X_test: {X_test.shape}")
display(f"Shape of y_train: {y_train.shape}")
display(f"Shape of y_test: {y_test.shape}")

"Original 'sex' categories: ['.' 'FEMALE' 'MALE']"

"Encoded 'sex' categories: [2 1 0]"

'Shape of X_train: (234, 4)'

'Shape of X_test: (101, 4)'

'Shape of y_train: (234,)'

'Shape of y_test: (101,)'

In [15]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

unique_labels = np.unique(y_test)

correct_target_names = le.inverse_transform(unique_labels)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, labels=unique_labels, target_names=correct_target_names)

display(f"Accuracy: {accuracy:.2f}")
display(f"Classification Report:\n{report}")

'Accuracy: 0.91'

'Classification Report:\n              precision    recall  f1-score   support\n\n      FEMALE       0.93      0.88      0.90        48\n        MALE       0.89      0.94      0.92        53\n\n    accuracy                           0.91       101\n   macro avg       0.91      0.91      0.91       101\nweighted avg       0.91      0.91      0.91       101\n'